In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from features_reindex import get_feature

In [30]:
root = '/itf-fi-ml/shared/users/ziyuzh/svm'
feature_info = dict()
feature_list= ['uniport_ppi_2019','ppi_2019_dw_40','uniport_ppi_2017','ppi_2017_dw_80','uniport_exp','uniport_bio','uniport_seq','uniport_esm']
merged_df = None
for feature in feature_list:
    feature_df = get_feature(root, feature)
    info_list = []

    feature_cols = [col for col in feature_df.columns if col.startswith('feature')]
    if feature_cols:
        scaler = MinMaxScaler()
        feature_matrix = feature_df[feature_cols].values
        info_list.append(feature_matrix.shape)
        info_list.append(round(np.linalg.norm(feature_matrix, ord=2),2))
        scaled_feature_matrix = scaler.fit_transform(feature_matrix)
        info_list.append(round(np.linalg.norm(scaled_feature_matrix, ord=2),2))
    feature_info[feature] = info_list

In [31]:
feature_info

{'uniport_ppi_2019': [(17222, 128), 123.64, 742.41],
 'ppi_2019_dw_40': [(17223, 128), 145.86, 741.57],
 'uniport_ppi_2017': [(17064, 128), 115.58, 741.3],
 'ppi_2017_dw_80': [(17065, 128), 139.25, 736.08],
 'uniport_exp': [(16586, 200), 150.92, 924.01],
 'uniport_bio': [(16941, 100), 400.82, 655.04],
 'uniport_seq': [(16677, 1024), 104.0, 2129.32],
 'uniport_esm': [(16673, 1280), 891.75, 2341.61]}

In [32]:
def get_intersected_info(year):
    feature_info_intersection = dict()
    merged_df = None
    if year == 2017:
        feature_list = ['uniport_ppi_2017','ppi_2017_dw_80','uniport_exp','uniport_seq','uniport_esm']
    elif year == 2019:
        feature_list = ['uniport_ppi_2019','ppi_2019_dw_40','uniport_bio','uniport_seq','uniport_esm']
    for feature in feature_list:
        feature_df = get_feature(root, feature)

        # Rename columns starting with 'feature'
        feature_df.rename(columns={
            col: f"{feature}_{col}" if col.startswith('feature') else col
            for col in feature_df.columns
        }, inplace=True)

        # Merge iteratively to avoid keeping all DataFrames
        if merged_df is None:
            merged_df = feature_df
        else:
            merged_df = pd.merge(merged_df, feature_df, on='string_id', how='inner')


    for feature_name in feature_list:
        info_list = []
        select_columns = [col for col in merged_df.columns if col.startswith(feature_name)]
        feature_matrix = merged_df[select_columns].values

        info_list.append(feature_matrix.shape)

        info_list.append(round(np.linalg.norm(feature_matrix, ord=2),2))

        scaled_feature_matrix = scaler.fit_transform(feature_matrix)
        info_list.append(round(np.linalg.norm(scaled_feature_matrix, ord=2),2))
        feature_info_intersection[feature_name] = info_list
    return feature_info_intersection


In [33]:
get_intersected_info(2017)

{'uniport_ppi_2017': [(15328, 128), 107.07, 700.76],
 'ppi_2017_dw_80': [(15328, 128), 128.43, 698.78],
 'uniport_exp': [(15328, 200), 141.16, 889.46],
 'uniport_seq': [(15328, 1024), 99.7, 2040.74],
 'uniport_esm': [(15328, 1280), 853.15, 2246.02]}

In [34]:
get_intersected_info(2019)

{'uniport_ppi_2019': [(15686, 128), 116.11, 706.8],
 'ppi_2019_dw_40': [(15686, 128), 137.74, 707.52],
 'uniport_bio': [(15686, 100), 386.02, 630.23],
 'uniport_seq': [(15686, 1024), 100.81, 2065.63],
 'uniport_esm': [(15686, 1280), 861.47, 2270.61]}